# llms

> Generic LLM-calling utilities (models, prompting, tool schemas) -- deliberately kept independent of boopiter's own Notebook/Cell model, so this module never imports from `cells.py`. `cells.py` imports from here, never the other way around.

In [ ]:
#| default_exp llms

In [ ]:
#| export
import inspect, os, re, time
from fastcore.utils import *
from fasthtml.common import Details, Summary, Ul, Li, Pre, to_xml
from lisette import *

## Tool selection

Which tool functions to offer the LLM. `_LOCAL_TOOLS` / `DEFAULT_TOOL_SELECTION` define the available sources and the default set; `get_tool_list` assembles the actual function list from whichever sources are selected.

In [ ]:
#| export
# slmn (github.com/drscotthawley/slmn, a separate general-purpose toolkit -- see
# boopiter/pyproject.toml for the git dependency) is imported as plain modules here, not
# re-exported via `import *` -- see get_tool_list() below, which assembles actual tool
# functions on demand instead of merging namespaces.
import slmn.nbtools as _slmn_nbtools
import slmn.misc as _slmn_misc
import slmn.remote as _slmn_remote
import slmn.dead_drop as _dd

# slmn.remote also has remote_launch/remote_status/remote_smoke_test, which can run arbitrary
# commands on a remote host over ssh -- a much bigger capability than the rest of slmn's tools.
# Least-privilege: don't hand those to the LLM by default, just the read-only/informational ones.
_SLMN_REMOTE_SAFE = ('fetch_url', 'check_ci', 'remote_gpu_free')

_LOCAL_TOOLS = []  # boopiter-defined tools always available, independent of slmn; empty for now, add as needed

In [ ]:
#| export
DEFAULT_TOOL_SELECTION = {'boopiter': True, 'slmn-nbtools': True, 'slmn-misc': True, 'slmn-remote': False}  # matches the wrench-icon Tools menu's default checkbox states

def get_tool_list(selection:dict=None # which tool sources to include -- keys 'boopiter'/'slmn-nbtools'/'slmn-misc'/'slmn-remote', bool values (see the wrench-icon Tools menu in the GUI). Defaults to DEFAULT_TOOL_SELECTION if omitted. 'slmn-remote', even when selected, only ever contributes its safe/read-only subset (_SLMN_REMOTE_SAFE) -- remote_launch/remote_status/remote_smoke_test (arbitrary remote command execution over ssh) are never included here, by design, regardless of selection.
                   ) -> list:
    "Assemble the list of tool functions to offer an LLM, from whichever sources are selected. Returns actual callables, not names -- pass straight to prompt_llm(tools=...). Per-notebook ad-hoc tools (see add_tool()) are layered on top of this by the caller, not included here."
    selection = selection if selection is not None else DEFAULT_TOOL_SELECTION
    tools = []
    if selection.get('boopiter'): tools += _LOCAL_TOOLS
    if selection.get('slmn-nbtools'): tools += [getattr(_slmn_nbtools, name) for name in _slmn_nbtools.__all__]
    if selection.get('slmn-misc'): tools += [getattr(_slmn_misc, name) for name in _slmn_misc.__all__]
    if selection.get('slmn-remote'): tools += [getattr(_slmn_remote, name) for name in _SLMN_REMOTE_SAFE]
    return tools

In [ ]:
#| export
OLLAMA_PREFIX = 'ollama_chat/'  # litellm has two routes to the same Ollama server: 'ollama/' (its /api/generate endpoint) and 'ollama_chat/' (/api/chat). Same server, same models -- but only the chat route's request builder puts 'keep_alive' at the top level of the request where Ollama reads it; the generate route sweeps every extra kwarg into a nested 'options' dict, where Ollama silently ignores it (measured: keep_alive='30s' still expired in the default 300s). See _chat_callkw().
OLLAMA_KEEP_ALIVE = '1m'  # how long Ollama holds a model in memory after answering, sent on every call (see _chat_callkw). Ollama's own default is 5 minutes; this shortens the window in which a model you've stopped using is still occupying RAM. Note this is a backstop, not the main mechanism -- switching models in the brain menu frees the old one immediately (see sync_ollama_loaded).
OLLAMA_URL = os.environ.get('OLLAMA_HOST') or 'http://localhost:11434'
if not OLLAMA_URL.startswith('http'): OLLAMA_URL = 'http://' + OLLAMA_URL  # OLLAMA_HOST is conventionally set bare ('127.0.0.1:11434'), but httpx needs a full URL

def ollama_loaded() -> list[str]:
    "Names of the models Ollama is currently holding in memory, per its /api/ps endpoint -- not to be confused with get_ollama_list()'s /api/tags, which lists every model on disk whether loaded or not. Returns [] rather than raising if Ollama is unreachable, so callers can treat 'no server' and 'nothing loaded' the same way."
    import httpx
    try: return [m['model'] for m in httpx.get(f'{OLLAMA_URL}/api/ps', timeout=5).json().get('models', [])]
    except Exception: return []

def unload_ollama_model(name:str) -> bool:
    "Evict one model from Ollama's memory, given its bare name ('qwen3:14b' -- NOT the 'ollama/'-prefixed id used everywhere else, see get_ollama_list()). Done over HTTP, as a generate request with keep_alive=0 (Ollama's documented way to drop a model immediately), rather than by shelling out to `ollama stop`: the CLI has to be on the *server process's* PATH -- which it often isn't when boopiter is launched from a desktop launcher or systemd unit rather than a login shell -- and it always talks to localhost, whereas this honours OLLAMA_URL like the rest of this module. Returns whether it worked, and warns (rather than failing silently) if not, since a silent no-op here looks exactly like a memory leak."
    import httpx, warnings
    try:
        httpx.post(f'{OLLAMA_URL}/api/generate', json={'model': name, 'keep_alive': 0}, timeout=60).raise_for_status()
        return True
    except Exception as e:
        warnings.warn(f"Couldn't unload Ollama model {name}: {e}")
        return False

def sync_ollama_loaded(keep:list) -> list[str]:
    "Enforce the invariant that *only* the models in `keep` stay resident in Ollama: unload every currently-loaded model that isn't one of them. Returns the names actually unloaded. `keep` accepts bare model names or 'ollama/'-prefixed ids interchangeably, and ignores both None and non-Ollama ids like 'deaddrop/...', so a caller can pass nb.standard_model/nb.reasoning_model straight through. Sweeping the live /api/ps list -- instead of remembering which model we last switched away from -- is the point: it also clears models left resident by a previous boopiter process, by another client of the same Ollama server, or by an earlier unload that quietly failed. It's idempotent, so calling it on every model-menu change costs one cheap GET when there's nothing to do."
    keep = {m.removeprefix(OLLAMA_PREFIX) for m in keep if m}
    return [m for m in ollama_loaded() if m not in keep and unload_ollama_model(m)]

## Listing available models

`get_ollama_list` reads every locally-available Ollama model from `/api/tags`; `get_model_list` wraps it (and any other sources) into a uniform list of model-info dicts.

In [ ]:
#| export
_CAPS_CACHE = {}  # digest -> capabilities list. A digest identifies exact model content, so its capabilities can never change under us; nothing here ever needs invalidating, and a re-pulled/retagged model simply arrives with a new digest and misses the cache.

def _ollama_caps(m:dict) -> list[str]:
    "The capability list ('vision'/'tools'/'thinking') for one /api/tags entry, memoized by digest in _CAPS_CACHE. This is the expensive half of get_ollama_list() -- ~86ms per model against a local Ollama, versus ~8ms for the whole /api/tags listing -- which is why it's cached: the brain menu re-lists models on every hover (see refresh_models() in cells.py), and re-probing unchanged models each time would put ~0.9s of pointless work behind a menu with ten models in it. A failed probe returns [] *without* caching, so a transient error doesn't permanently strip a model of its capabilities (which would, e.g., silently drop it from the reasoning-model picker until the next server restart)."
    import httpx
    key = m.get('digest') or m['model']  # digest preferred; the name is a weaker fallback for an Ollama old enough not to report one
    if key in _CAPS_CACHE: return _CAPS_CACHE[key]
    try: caps = httpx.post(f"{OLLAMA_URL}/api/show", json={'model': m['model']}, timeout=30).json().get('capabilities', [])
    except Exception: return []
    _CAPS_CACHE[key] = caps
    return caps

def get_ollama_list(strict:bool=False) -> list[dict]:
    "Get info on every locally-available Ollama model: the raw /api/tags entry (name, details incl. parameter_size/family) plus an added 'id' key ('ollama/<model>', the string used elsewhere as the model identifier -- see nb.model) and a 'capabilities' list (e.g. 'vision'/'tools'/'thinking'). Capabilities are NOT in /api/tags -- only the per-model /api/show endpoint reports them (see _ollama_caps, which caches them and is why calling this repeatedly is cheap). Those probes are independent per model and each is almost entirely network wait, so cache misses are fetched on a small thread pool rather than in series -- with a cold cache that turns N sequential round trips into roughly one. Returns [] and warns if Ollama is unavailable -- unless `strict`, which raises instead, so a caller that already has a good listing (refresh_models(), re-checking on every brain-menu hover) can tell 'Ollama is down' apart from 'Ollama has no models' and keep showing what it had rather than blanking the menu on a transient hiccup. A model whose individual capability probe fails just gets an empty capability list, not a failed listing."
    import httpx, warnings
    from concurrent.futures import ThreadPoolExecutor
    try: models = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=10).json().get('models', [])
    except Exception as e:
        if strict: raise
        warnings.warn(f"Ollama not available: {e}")
        return []
    for m in models: m['id'] = OLLAMA_PREFIX + m['model']
    models.sort(key=lambda m: m['model'].lower())  # /api/tags returns models in an order that is NOT stable between calls -- sort so the dropdown doesn't silently reshuffle itself, and so refresh_models() can compare two listings and actually tell whether anything changed
    if models:
        with ThreadPoolExecutor(max_workers=min(8, len(models))) as ex:
            for m, caps in zip(models, ex.map(_ollama_caps, models)): m['capabilities'] = caps
    return models

In [ ]:
#| export
def get_model_list(strict:bool=False) -> list[dict]:
    "Wrapper routine to get info dicts (see get_ollama_list()) for all available models from all sources. Live dead-drop sessions (see _deaddrop_sessions) are listed as models too, one 'deaddrop/<session-id>' entry each: selecting one in the brain-menu dropdown is what binds this notebook's Prompt cells to that session's inbox/outbox. Modelling the binding as a model choice -- rather than as a global set once per process -- means it's visible in the UI, saved with the notebook, and per-notebook, so several notebooks on different topics can each talk to their own session without crossing wires, and a restart can't silently reroute one of them. 'deaddrop/claude' remains the original single-file prompts/ + responses/ route."
    deaddrop = [{'id': 'deaddrop/claude', 'model': 'claude', 'capabilities': ['vision', 'thinking']}]
    deaddrop += [{'id': f'deaddrop/{s}', 'model': s, 'capabilities': ['vision', 'thinking']}
                 for s in _deaddrop_sessions()]
    return get_ollama_list(strict=strict) + deaddrop  # TODO: add more model source, e.g. cloud, fileio


In [ ]:
#| eval: false
get_model_list()

['ollama/qwen2.5-coder:latest',
 'ollama/gemma3:4b',
 'ollama/llama3.1:latest',
 'ollama/qwen2.5:latest']

## Forcing streaming responses

A monkeypatch (`_call`, over the saved `_orig_chat_call`) makes the chat client always yield streamed responses, working around a limitation in the underlying library.

In [ ]:
#| export
# lisette + local (Ollama) models: passing non-empty `tools=` combined with `tool_choice='none'`
# triggers a bug in litellm's MCP-handler codepath that returns a raw dict instead of a proper
# response object (AttributeError: 'dict' object has no attribute 'choices'). Same issue hit by
# SBrewer15/CellMate (https://github.com/SBrewer15/CellMate) -- this is their patch, adopted as-is:
# drop tool_schemas for just that one call whenever tool_choice=='none', then restore them after.
_orig_chat_call = Chat._call


In [ ]:
#| export
@patch
def _call(self:Chat, msg:str|None=None, prefill:str|None=None, temp:float|None=None, think:str|None=None,
          search:str|None=None, stream:bool=False, max_steps:int=2, step:int=1, final_prompt:dict|None=None,
          tool_choice:str|None=None, max_tokens:int|None=None, **kwargs):
    "Internal method that always yields responses -- patched (see the comment above) to avoid a litellm/Ollama tool-calling bug."
    _orig_tools = self.tool_schemas
    if tool_choice == 'none': self.tool_schemas, tool_choice = None, None
    try: yield from _orig_chat_call(self, msg, prefill, temp, think, search, stream, max_steps, step, final_prompt, tool_choice, max_tokens, **kwargs)
    finally: self.tool_schemas = _orig_tools

In [ ]:
#| export
def _details_block(rows:list[tuple[str,str]], reasoning:str|None=None) -> str:
    "Build the collapsible <details> block of LLM call metadata (e.g. model/finish reason/tokens/tool calls) shown above a reply -- shared by the real-API path (_reply_details_html, below) and the dead_drop path (_deaddrop_stream_reply), so there's one escaped-HTML builder instead of two hand-written copies. Built from FT components (not an f-string), so every interpolated value -- including a dead_drop responder's self-reported model name, which is model-generated text, not app-controlled -- is HTML-escaped automatically rather than spliced in raw. Not part of the reply's actual content, kept out of Cell.source (see Cell.details) so it's never sent back to the model as context, and shown collapsed, in gray, above the real text. onclick=stopPropagation keeps a click on <summary> from also bubbling into the cell's click-anywhere-to-edit handler."
    items = [Li(f'{k}: {v}') for k, v in rows]
    kids = [Summary('Reply details'), Ul(*items)]
    if reasoning:
        kids.append(Pre(reasoning, style='white-space:pre-wrap'))
    return to_xml(Details(*kids, cls='text-gray-400', onclick='event.stopPropagation()'), indent=False)

In [ ]:
#| export
def _reply_details_html(response, msg) -> str:
    "The <details> block (see _details_block) for a real API call's response: model, finish reason, token counts, tool calls, and reasoning content, if any."
    u = response.usage
    rows = [('Model', response.model), ('Finish reason', response.choices[0].finish_reason)]
    if u: rows.append(('Tokens', f'{u.prompt_tokens} prompt + {u.completion_tokens} completion = {u.total_tokens} total'))
    if msg.tool_calls: rows.append(('Tool calls', ', '.join(tc.function.name for tc in msg.tool_calls)))
    return _details_block(rows, reasoning=getattr(msg, 'reasoning_content', None))

## Reply metadata

`_details_block` builds the collapsible `<details>` panel of call metadata (model, finish reason, token counts, tool calls); `_reply_details_html` fills it in for a real API response. Rendered with escaped FT components, not raw HTML.

### Why tools aren't passed as native API `tools=`

Early on, enabling any tool source made local models (tested worst-case: `qwen2.5-coder:latest`, but reproduced up through `qwen3.6:27b`) reflexively try to call a tool on *every* prompt, including plain "write me some code" requests that had nothing to do with any tool -- producing garbled, empty, or JSON-as-prose replies instead of a normal answer.

The root cause: Ollama/Qwen's chat template wraps any non-empty `tools=` API parameter into a `<tools>...</tools>` special-token block. That structural block biases these models into feeling obligated to fill a `<tool_call>` slot, regardless of `tool_choice` (`'auto'` vs. the default `None` made no measurable difference) and regardless of a system prompt saying "only use a tool if you actually need one" (that improved *presentation* -- coherent prose vs. raw JSON -- but didn't stop the reflex). Bigger models weren't immune either, just failed differently (e.g. the tool call landing in `reasoning_content` while `content` came back empty).

The fix, found by reading how SolveIt's `dialoghelper`/`pyskills` solve the same problem: never populate the native `tools=` parameter. Instead, describe the available functions in the system prompt as plain Python callables already present in the execution namespace (see `tools_system_prompt()` below), and let the model express "I need a tool" by writing an ordinary ```python code block as part of its reply -- the same code-generation skill these models are already reliably good at, rather than their much less reliable native function-calling judgment. `stream_llm_reply()` (and `_push_tools()` in `cells.py`, which keeps those functions live in the shared kernel namespace) implement that pattern end to end.

In [ ]:
#| export
def _tool_doc_line(fn) -> str:
    "One-line description of a tool function for the system prompt: name, signature, and the first line of its docstring."
    sig = str(inspect.signature(fn))
    doc = (inspect.getdoc(fn) or '').split('\n')[0]
    return f"- {fn.__name__}{sig}: {doc}"

def tools_system_prompt(tools:list|None) -> str:
    "Build a system-prompt block describing `tools` as plain Python functions already available in the model's execution environment -- NOT passed as native API tool schemas. This sidesteps a real reliability problem found via testing: Ollama/Qwen's chat template wraps any non-empty `tools=` into a `<tools>` special-token block that biases even capable local models into reflexive, often-wrong tool calls, regardless of tool_choice or how carefully the system prompt says 'only if needed' (both were tried and didn't help). Routing tool use through the model's own code-writing judgment instead -- which local models are much more reliably good at -- fixed it in practice. Same approach dialoghelper/pyskills use for SolveIt (not a dependency here, just the inspiration). Returns '' if `tools` is empty, so no system prompt is added at all when there's nothing to describe."
    if not tools: return ''
    lines = '\n'.join(_tool_doc_line(fn) for fn in tools)
    return (
        "The following Python functions are already available in your code execution environment "
        "(no import needed) if a task genuinely requires one:\n"
        f"{lines}\n\n"
        "To use one, write ordinary Python code calling it inside a fenced ```python code block, "
        "as part of your normal reply -- the user can run that code directly. Do not describe a "
        "tool call as JSON or any other structured format; just write real Python code, and only "
        "when a task actually needs it. For requests that don't require reading/editing files or "
        "other tool-backed actions -- e.g. 'write me some code', general questions -- just answer "
        "directly; these helper functions are irrelevant to those."
    )

def _chat_callkw(model:str) -> dict:
    "The kwargs handed to litellm on every Chat() call for a given model. _skip_mcp_handler avoids litellm's MCP-proxy import chain (needs fastapi/orjson) that we don't use -- drop it (and re-add fastapi/orjson to pyproject.toml) if/when we actually want MCP tool support. Ollama models additionally carry keep_alive (see OLLAMA_KEEP_ALIVE), which only reaches Ollama at all because model ids use the 'ollama_chat/' route -- see OLLAMA_PREFIX. It's omitted for anything else, since it's an Ollama-specific field no other provider would know what to do with."
    kw = {'_skip_mcp_handler': True}
    if model.startswith('ollama'): kw['keep_alive'] = OLLAMA_KEEP_ALIVE
    return kw

def prompt_llm(context:str, model:str=OLLAMA_PREFIX+'qwen2.5-coder:latest', tools:list|None=None, think:str|None=None) -> tuple[str,str]:
    "Send a prompt to the LLM; returns (content, details_html) -- the reply text itself, and a separate collapsible <details> block of call metadata (model/tokens/finish reason/reasoning) meant to be stored apart from the reply (see Cell.details), not mixed into it. `tools`, if given, are described via tools_system_prompt() (code-callable, not native tool-calling) -- see there for why. `think`, if given ('l'/'m'/'h'), is lisette's own reasoning-effort control -- passed straight through to litellm, which maps it to Ollama's 'think' request field; only meaningful for a model that actually supports thinking (see the brain-icon reasoning-model picker)."
    chat = Chat(model, sp=tools_system_prompt(tools), tools=[], callkw=_chat_callkw(model)) # FYI: this makes a fresh stateless context each time. is that what we want?
    response = chat(context, think=think)
    msg = contents(response)
    return msg.content, _reply_details_html(response, msg)

_DEADDROP_DIR = Path.home() / 'dead_drop'  # holds the legacy prompts/ + responses/ pair, plus one subdirectory per session -- see slmn.dead_drop
_DEADDROP_SESSION_SUBDIRS = ('inbox', 'outbox')  # a session is a directory holding these two; prompts go in, replies come back out, and nothing is ever moved between them
_HAS_SESSIONS = hasattr(_dd, 'follow_reply')  # session routing is newer than the single-file exchange; without it, session models simply don't appear and 'deaddrop/claude' still works

def _deaddrop_sessions(dir=None) -> list[str]:
    "The dead-drop session ids that can currently be prompted into. Thin wrapper over slmn's list_sessions(require=...) -- the `require` filter is what keeps the legacy 'prompts'/'responses' directories, which sit in the same root and are not sessions, from being offered in the model dropdown as something you could address but never get an answer from. Returns [] on an installed slmn predating session routing."
    if not _HAS_SESSIONS: return []
    return _dd.list_sessions(str(dir or _DEADDROP_DIR), require=_DEADDROP_SESSION_SUBDIRS)

def _deaddrop_events(chunks, model:str, extra_rows:list=None):
    "Adapt slmn's plain text-chunk generator (follow_file/follow_reply) to the ('delta', ...) / ('final', ...) contract stream_llm_reply's callers expect. Everything protocol-shaped -- waiting for the file, tailing it as it grows, recognising the end-of-reply marker -- lives in slmn.dead_drop; what's left here is boopiter's own presentation concern: accumulate the chunks, then pull out the '---MODEL: ...---' line the responder was asked to add (see _DEADDROP_MODEL_INSTRUCTION) so the details block can name the model that actually answered rather than the generic 'deaddrop/...', and trim it off the visible reply. That name is untrusted, model-generated text -- a human is relaying whatever the other side typed -- so _details_block escapes it, exactly as on the real-API path."
    parts = []
    for chunk in chunks:
        parts.append(chunk)
        yield ('delta', chunk)
    content = ''.join(parts).rstrip()
    m = _DEADDROP_MODEL_RE.search(content)
    responder_model = (m.group(1) if m else model) + ' (dead drop)'
    if m: content = content[:m.start()].rstrip()
    yield ('final', content, _details_block([('Model', responder_model), *(extra_rows or [])]))

_DEADDROP_MODEL_INSTRUCTION = (
    "\n\n[system: immediately before the final '---DONE---' line, on its own line, include exactly "
    "one line in this format -- ---MODEL: <your model name and version>--- -- e.g. "
    "---MODEL: Claude Opus 4.8---. This identifies which model actually answered; it is parsed "
    "out and shown separately, not part of the visible reply.]"
)
_DEADDROP_MODEL_RE = re.compile(r'\n?---MODEL:\s*(.+?)\s*---\s*$')

def _deaddrop_stream_legacy(context:str, model:str, poll_interval:float=0.25):
    "The original single-file dead-drop route, used by the 'deaddrop/claude' model and kept for the hand-driven prompts/ + responses/ workflow (session models go through _deaddrop_stream_session instead): drop `context` (plus `_DEADDROP_MODEL_INSTRUCTION`, asking the responder to self-identify) as one auto-named prompt file, then follow the paired response file of the same name under _DEADDROP_DIR/responses. Identical in shape to the session route -- both hand a slmn text-chunk generator to _deaddrop_events -- differing only in where the two files live and in addressing no particular session; the 'model' on the other end is a human relaying a real Claude session through files."
    path = _dd.drop(str(_DEADDROP_DIR / 'prompts'), context + _DEADDROP_MODEL_INSTRUCTION)
    resp_path = _DEADDROP_DIR / 'responses' / Path(path).name
    yield from _deaddrop_events(_dd.follow_file(str(resp_path), poll_interval), model)

def _deaddrop_stream_session(context:str, model:str, session_id:str, poll_interval:float=0.25):
    "Route a Prompt-cell reply to one specific dead-drop session: drop the prompt as a bundle into that session's inbox/ (slmn's drop_prompt -- a directory written via one atomic rename, so a prompt can carry attachments alongside its text and a watcher never sees it half-written), then follow the reply streaming back out of its outbox/ (slmn's follow_reply). Just the two directories, with nothing ever moved between them: the reply's own '---DONE---' terminator says when it's finished, so there's no claimed/completed bookkeeping to keep in sync -- and unlike a reply delivered as an atomically-renamed bundle, it can be read while it's still being written, which is what lets the answer fill into the cell live."
    prompt_id = _dd.drop_prompt(str(_DEADDROP_DIR), session_id, {'prompt.md': context + _DEADDROP_MODEL_INSTRUCTION})
    chunks = _dd.follow_reply(str(_DEADDROP_DIR), session_id, prompt_id, poll_interval)
    yield from _deaddrop_events(chunks, model, [('Session', session_id), ('Prompt id', prompt_id)])

def _deaddrop_stream_reply(context:str, model:str, poll_interval:float=0.25):
    "Dispatch a 'deaddrop/...' model to the right dead-drop route. 'deaddrop/claude' is the original single-file prompts/ + responses/ exchange (_deaddrop_stream_legacy); anything else is read as a session id and addressed through that session's inbox/outbox (_deaddrop_stream_session). Picking the session from the model dropdown is what binds a notebook to it -- see _deaddrop_sessions/get_model_list -- so the binding is visible, per-notebook, and saved with the notebook, rather than living in process-wide state that a restart would silently drop and reroute."
    session_id = model.split('/', 1)[1] if '/' in model else 'claude'
    if session_id == 'claude':
        yield from _deaddrop_stream_legacy(context, model, poll_interval)
    elif not _HAS_SESSIONS:
        raise RuntimeError(f"Model {model!r} needs slmn's dead-drop session routing (drop_prompt/follow_reply), which this installed slmn doesn't have. Update slmn, or use 'deaddrop/claude'.")
    else:
        yield from _deaddrop_stream_session(context, model, session_id, poll_interval)

def stream_llm_reply(context:str, model:str, tools:list|None=None, think:str|None=None, images:list|None=None):
    "Generator streaming a model reply token-by-token, using the code-callable tool strategy (see tools_system_prompt) instead of native tool-calling. `images` is an optional list of raw image bytes to send alongside the context -- pass them only for models that advertise the vision capability (the caller checks; see cells.py's _start_prompt_run), as non-vision models error or ignore them. They ride as extra message parts via lisette's mk_msg (bytes -> base64 data URL), with the context text last. The dead-drop route ignores them for now -- file-based image handoff is its own upcoming feature. Yields ('delta', text) chunks as they arrive, then a final ('final', content, details_html) tuple once the response completes. Callers (e.g. cells.py's _run_prompt_bg) drive this into their own background-thread/UI state -- that part isn't LLM-specific, so it stays out of this module. `think` ('l'/'m'/'h' or None) -- see prompt_llm(). A `model` starting with 'deaddrop/' is routed through _deaddrop_stream_reply() instead of a real API call -- everything else about this function's contract (the yielded tuple shapes) is identical either way, so no caller needs to know or care which one answered."
    if model.startswith('deaddrop/'):
        yield from _deaddrop_stream_reply(context, model)
        return
    chat = Chat(model, sp=tools_system_prompt(tools), tools=[], callkw=_chat_callkw(model))
    content = [*images, context] if images else context  # image bytes first, question last -- and not named `msg`, which the final-chunk branch below reuses
    for chunk in chat(content, stream=True, think=think):
        if hasattr(chunk.choices[0], 'message'):  # the final item -- a full ModelResponse, not a delta
            msg = contents(chunk)
            yield ('final', msg.content, _reply_details_html(chunk, msg))
        else:
            delta = chunk.choices[0].delta.content
            if delta: yield ('delta', delta)

In [ ]:
#| export
_PREFERRED_MODEL_SUBSTR = 'qwen2.5-coder'  # used if present, regardless of exact tag/version


In [ ]:
#| eval: false
s ="Today is July 18. Who's one famous person with this birthday?"
c = prompt_llm(s) 
print(str(c[0]))
s = """
    Tell me the previous question I asked you, from the previous prompt. 
    I want to see if you retain state between calls"""
c = prompt_llm(s) 
print(str(c[0]))

One famous person born on July 18th is Mark Zuckerberg, the co-founder and CEO of Facebook.
I'm sorry for any confusion, but as an AI language model, I don't have the capability to remember or retain information across separate interactions. Each response is generated independently based on the input provided in each session. If you have a specific question or need assistance with something particular, feel free to ask!


### Tool Use

Example tool from lisette docs:

In [ ]:
def add_numbers(
    a: int,  # First number to add
    b: int   # Second number to add  
) -> int:
    "Add two numbers together"
    return a + b

In [ ]:
#| eval: false
res = prompt_llm("What's 47 + 23? Use the tool.", tools=[add_numbers])
print(res[0])

The sum of 47 and 23 is 70. I have completed my task as requested. If you need further assistance or have additional questions, feel free to ask!


In [ ]:
#| eval: false
# Confirms get_tool_list() assembles real, callable tools according to a selection dict.
from boopiter.llms import get_tool_list, DEFAULT_TOOL_SELECTION
print(len(get_tool_list()), "tools with defaults:", [t.__name__ for t in get_tool_list()])
print(len(get_tool_list({'boopiter': True, 'slmn-nbtools': False, 'slmn-misc': False, 'slmn-remote': True})), "tools with only boopiter+remote")